# Task 3 - Effect Characterization of Adversarial Depth Patches

Quantifies the effectiveness of the Task 2 adversarial patch across scenes,
object types, and viewing conditions. Also trains 2 tuning variants (V3a larger
patch, V3b stronger lambda_adv) to assess whether Task 2's v2 saturation was
architecture- or hyperparameter-limited.


## 1. Imports

In [ ]:
import os
import io
import csv
import copy
import json
import time
import hashlib
import random
import unittest
from pathlib import Path
from collections import defaultdict
import glob as glob_module
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

In [ ]:
import transformers
print(f'PyTorch:      {torch.__version__}')
print(f'CUDA avail:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:          {torch.cuda.get_device_name(0)}')
    print(f'VRAM:         {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')
print(f'Transformers: {transformers.__version__}')
assert torch.cuda.is_available(), 'CUDA required for attack loop on RTX 4060 Ti.'

PyTorch:      2.6.0+cu124
CUDA avail:   True
GPU:          NVIDIA GeForce RTX 4060 Ti
VRAM:         8.59 GB
Transformers: 5.5.3


## 2. Globals

In [ ]:

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_CHECKPOINT = 'depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf'
DATA_ROOT = r'D:\cv\data\nyu_v2'
CHECKPOINT_DIR = r'D:\cv\checkpoints'
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
BASELINE_METRICS_PATH = os.path.join(CHECKPOINT_DIR, 'baseline_metrics.json')
VIZ_DIR = os.path.join(CHECKPOINT_DIR, 'viz_task2')
os.makedirs(VIZ_DIR, exist_ok=True)
INPUT_SIZE = 518
NYU_HEIGHT = 480
NYU_WIDTH  = 640
EVAL_CROP_TOP, EVAL_CROP_BOTTOM = 45, 471
EVAL_CROP_LEFT, EVAL_CROP_RIGHT = 41, 601
MIN_DEPTH = 1e-3
MAX_DEPTH = 10.0
PATCH_SIZE = 128
PATCH_INIT = 'uniform'
PATCH_PATH = os.path.join(CHECKPOINT_DIR, 'adv_patch.pt')
PATCH_PNG  = os.path.join(CHECKPOINT_DIR, 'adv_patch.png')
REPORT_PATH = os.path.join(CHECKPOINT_DIR, 'attack_report.json')
TARGET_DEPTH = MAX_DEPTH
MASK_DILATE_PX = 3
ADV_LOSS_EPS = 1e-3
GRAD_CLIP_PATCH = 1.0
EOT_ROT_DEG    = 15.0
EOT_SCALE_MIN, EOT_SCALE_MAX = 0.7, 1.3
EOT_TRANS_FRAC = 0.20
EOT_BRIGHT     = 0.20
EOT_CONTRAST_MIN, EOT_CONTRAST_MAX = 0.8, 1.2
EOT_NOISE_STD  = 0.02
LAMBDA_ADV = 1.0
LAMBDA_NPS = 5e-3
LAMBDA_TV  = 1e-3
ATTACK_LR    = 1e-2
N_STEPS      = 3000
ATTACK_BATCH = 4
LOG_EVERY    = 25
SAVE_EVERY   = 250
USE_AMP_FORWARD = True
TRAIN_SUBSET_SIZE = 15000
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
PRINTABLE_COLORS = torch.tensor([
    [0.098, 0.082, 0.082], [0.173, 0.102, 0.094], [0.290, 0.118, 0.106],
    [0.392, 0.141, 0.137], [0.510, 0.165, 0.169], [0.627, 0.216, 0.212],
    [0.722, 0.302, 0.286], [0.098, 0.106, 0.204], [0.102, 0.153, 0.345],
    [0.106, 0.196, 0.467], [0.106, 0.243, 0.588], [0.102, 0.298, 0.706],
    [0.122, 0.357, 0.796], [0.235, 0.431, 0.820], [0.102, 0.314, 0.129],
    [0.106, 0.408, 0.165], [0.106, 0.502, 0.200], [0.114, 0.596, 0.243],
    [0.204, 0.694, 0.322], [0.341, 0.769, 0.427], [0.761, 0.694, 0.098],
    [0.827, 0.769, 0.137], [0.878, 0.835, 0.192], [0.914, 0.894, 0.263],
    [0.953, 0.949, 0.380], [0.996, 0.996, 0.996], [0.855, 0.855, 0.855],
    [0.627, 0.627, 0.627], [0.392, 0.392, 0.392], [0.157, 0.157, 0.157],
], dtype=torch.float32)
print(f'DEVICE:       {DEVICE}')
print(f'MODEL:        {MODEL_CHECKPOINT}')
print(f'PATCH_SIZE:   {PATCH_SIZE}x{PATCH_SIZE} -> {PATCH_SIZE*PATCH_SIZE} px, {100*PATCH_SIZE**2/INPUT_SIZE**2:.2f}% of 518^2 frame')
print(f'TARGET_DEPTH: {TARGET_DEPTH} m (push-farther / erase)')
print(f'ATTACK:       bs={ATTACK_BATCH} lr={ATTACK_LR} steps={N_STEPS}')
print(f'PALETTE:      {PRINTABLE_COLORS.shape[0]} printable colors')
print(f'SEED:         {SEED}')


DEVICE:       cuda
MODEL:        depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf
PATCH_SIZE:   128x128 -> 16384 px, 6.11% of 518^2 frame
TARGET_DEPTH: 10.0 m (push-farther / erase)
ATTACK:       bs=4 lr=0.01 steps=3000
PALETTE:      30 printable colors
SEED:         42


In [ ]:
T3_VIZ_DIR     = os.path.join(CHECKPOINT_DIR, 'viz_task3')
T3_REPORT_PATH = os.path.join(CHECKPOINT_DIR, 'task3_report.json')
os.makedirs(T3_VIZ_DIR, exist_ok=True)
PATCH_V2_PATH = os.path.join(CHECKPOINT_DIR, 'adv_patch.pt')
PATCH_V1_PATH = os.path.join(CHECKPOINT_DIR, 'adv_patch_v1_L1_96.pt')
ATTACK_REPORT_V2_PATH = os.path.join(CHECKPOINT_DIR, 'attack_report.json')
SWEEP_GRID = {
    'rotation':    [-30, -15, -5, 0, 5, 15, 30],
    'scale':       [0.5, 0.7, 1.0, 1.3, 1.5],
    'translation': [0.0, 0.1, 0.2, 0.3],
    'brightness':  [-0.30, -0.15, 0.0, 0.15, 0.30],
    'contrast':    [0.6, 0.8, 1.0, 1.2, 1.4],
    'noise_std':   [0.00, 0.02, 0.05, 0.10],
}
SWEEP_SUBSET_SIZE = 96
CROSS_SUBSET_SIZE = 64
SWEEP_SEED        = 42
CROSS_SEED        = 43
POSITION_GRID_N = 3
POSITION_OFFSETS = [-0.30, 0.0, 0.30]
DEPTH_BUCKETS = {
    'shallow': (0.0, 2.0),
    'mid':     (2.0, 4.0),
    'deep':    (4.0, 10.0),
}
SCENE_CATEGORIES = ['bedroom', 'living_room', 'kitchen', 'bathroom',
                    'office', 'dining_room']
PER_SCENE_SAMPLES = 100
PER_SCENE_SEED    = 44
VARIANT_CONFIGS = {
    'v3a_large192': {
        'patch_size': 192, 'lambda_adv': 1.0, 'lambda_nps': 5e-3,
        'lambda_tv': 1e-3, 'target_depth': 10.0, 'adv_loss_eps': 1e-3,
        'n_steps': 3000, 'attack_lr': 1e-2,
    },
    'v3b_ladvx5': {
        'patch_size': 128, 'lambda_adv': 5.0, 'lambda_nps': 2e-3,
        'lambda_tv': 1e-3, 'target_depth': 10.0, 'adv_loss_eps': 1e-3,
        'n_steps': 3000, 'attack_lr': 1e-2,
    },
}
print(f'T3_VIZ_DIR:   {T3_VIZ_DIR}')
print(f'Sweep subset: {SWEEP_SUBSET_SIZE} imgs (seed {SWEEP_SEED})')
print(f'Cross subset: {CROSS_SUBSET_SIZE} imgs (seed {CROSS_SEED})')
print(f'Position grid: {POSITION_GRID_N}x{POSITION_GRID_N} (offsets {POSITION_OFFSETS})')
print(f'Depth buckets: {list(DEPTH_BUCKETS.keys())}')
print(f'Scene cats:    {SCENE_CATEGORIES}')
print(f'Variants:      {list(VARIANT_CONFIGS.keys())}')


T3_VIZ_DIR:   D:\cv\checkpoints\viz_task3
Sweep subset: 96 imgs (seed 42)
Cross subset: 64 imgs (seed 43)
Position grid: 3x3 (offsets [-0.3, 0.0, 0.3])
Depth buckets: ['shallow', 'mid', 'deep']
Scene cats:    ['bedroom', 'living_room', 'kitchen', 'bathroom', 'office', 'dining_room']
Variants:      ['v3a_large192', 'v3b_ladvx5']


In [ ]:
class TestT3Globals(unittest.TestCase):
    def test_v2_patch_exists(self):
        self.assertTrue(os.path.exists(PATCH_V2_PATH), f'missing {PATCH_V2_PATH}')
    def test_attack_report_v2_exists(self):
        self.assertTrue(os.path.exists(ATTACK_REPORT_V2_PATH))
    def test_sweep_grid_nonempty(self):
        for k, v in SWEEP_GRID.items():
            self.assertGreaterEqual(len(v), 3, f'{k} needs >= 3 values')
    def test_subset_sizes(self):
        self.assertLessEqual(SWEEP_SUBSET_SIZE, 654)
        self.assertLessEqual(CROSS_SUBSET_SIZE, SWEEP_SUBSET_SIZE)
    def test_bucket_boundaries_contiguous(self):
        b = sorted(DEPTH_BUCKETS.items(), key=lambda kv: kv[1][0])
        for i in range(len(b) - 1):
            self.assertAlmostEqual(b[i][1][1], b[i+1][1][0])
        self.assertEqual(b[0][1][0], 0.0)
        self.assertAlmostEqual(b[-1][1][1], MAX_DEPTH)
    def test_variant_configs_valid(self):
        for name, cfg in VARIANT_CONFIGS.items():
            for key in ('patch_size','lambda_adv','lambda_nps','lambda_tv',
                        'target_depth','adv_loss_eps','n_steps','attack_lr'):
                self.assertIn(key, cfg, f'{name} missing {key}')
    def test_position_offsets_symmetric(self):
        self.assertAlmostEqual(POSITION_OFFSETS[0], -POSITION_OFFSETS[-1])
unittest.main(argv=['_'], module='__main__', exit=False, verbosity=2)


test_attack_report_v2_exists (__main__.TestT3Globals.test_attack_report_v2_exists) ... 

ok


test_bucket_boundaries_contiguous (__main__.TestT3Globals.test_bucket_boundaries_contiguous) ... 

ok


test_position_offsets_symmetric (__main__.TestT3Globals.test_position_offsets_symmetric) ... 

...

test_v2_patch_exists (__main__.TestT3Globals.test_v2_patch_exists) ... 

ok


test_variant_configs_valid (__main__.TestT3Globals.test_variant_configs_valid) ... 

ok


----------------------------------------------------------------------
Ran 7 tests in 0.004s

OK


## 3. Utils

### 3.1 Reused from Task 1/2 (verbatim)

These cells are lifted unchanged from Task 2's `_task2_cells.py`.
They define the depth-metric computation, NYU data loading, losses, EoT
pipeline, and patch composition used throughout Task 3.

In [ ]:
class SILogLoss(nn.Module):
    """Scale-Invariant Logarithmic Loss (Eigen 2014). Reused from Task 1."""
    def __init__(self, variance_focus: float = 0.5):
        super().__init__()
        self.variance_focus = variance_focus
    def forward(self, pred, gt, valid_mask=None):
        if valid_mask is None:
            valid_mask = (gt > MIN_DEPTH) & (gt < MAX_DEPTH)
        valid_mask = valid_mask.bool()
        pred_v = pred[valid_mask].clamp(min=MIN_DEPTH)
        gt_v   = gt[valid_mask].clamp(min=MIN_DEPTH)
        if pred_v.numel() == 0:
            return torch.tensor(0.0, device=pred.device, requires_grad=True)
        d = torch.log(pred_v) - torch.log(gt_v)
        return torch.sqrt(torch.mean(d ** 2) - self.variance_focus * (torch.mean(d) ** 2) + 1e-12)

In [ ]:
def compute_depth_metrics(pred, gt, min_depth=MIN_DEPTH, max_depth=MAX_DEPTH, eigen_crop: bool = False):
    """8 standard NYU depth metrics. Reused from Task 1."""
    pred = pred.detach().float()
    gt = gt.detach().float()
    if eigen_crop:
        pred = pred[..., EVAL_CROP_TOP:EVAL_CROP_BOTTOM, EVAL_CROP_LEFT:EVAL_CROP_RIGHT]
        gt   = gt[...,   EVAL_CROP_TOP:EVAL_CROP_BOTTOM, EVAL_CROP_LEFT:EVAL_CROP_RIGHT]
    valid = (gt > min_depth) & (gt < max_depth)
    pred_v = pred[valid].clamp(min=min_depth, max=max_depth)
    gt_v = gt[valid]
    if gt_v.numel() == 0:
        return {k: 0.0 for k in ('abs_rel','sq_rel','rmse','rmse_log','log10','delta1','delta2','delta3')}
    abs_diff = (pred_v - gt_v).abs()
    ratio = torch.max(pred_v / gt_v, gt_v / pred_v)
    return {
        'abs_rel':  (abs_diff / gt_v).mean().item(),
        'sq_rel':   ((abs_diff ** 2) / gt_v).mean().item(),
        'rmse':     torch.sqrt((abs_diff ** 2).mean()).item(),
        'rmse_log': torch.sqrt(((torch.log(pred_v) - torch.log(gt_v)) ** 2).mean()).item(),
        'log10':    (torch.log10(pred_v) - torch.log10(gt_v)).abs().mean().item(),
        'delta1':   (ratio < 1.25   ).float().mean().item(),
        'delta2':   (ratio < 1.25**2).float().mean().item(),
        'delta3':   (ratio < 1.25**3).float().mean().item(),
    }

In [ ]:
def load_depth_map(path, max_depth=MAX_DEPTH):
    depth_raw = np.array(Image.open(path))
    if depth_raw.dtype == np.uint8:
        return depth_raw.astype(np.float32) / 255.0 * max_depth
    return depth_raw.astype(np.float32) / 1000.0
def verify_gradient_flow(model, sample_input, device=None):
    device = device or DEVICE
    model.eval()
    x = sample_input.clone().detach().to(device).requires_grad_(True)
    out = model(pixel_values=x)
    pred = out.predicted_depth if hasattr(out, 'predicted_depth') else out[0]
    pred.sum().backward()
    ok = (x.grad is not None) and (not torch.all(x.grad == 0))
    if ok:
        g = x.grad.abs()
        print(f'  input grad: min={g.min():.2e} max={g.max():.2e} mean={g.mean():.2e}')
    else:
        print('  WARNING: no gradient flow to input!')
    model.zero_grad()
    return ok


In [9]:
def discover_nyu_pairs(data_root):
    """NYU v2 Kaggle soumikrakshit layout discovery. Reused from Task 1."""
    data_root = str(data_root)
    csv_files = sorted(glob_module.glob(os.path.join(data_root, '**', '*.csv'), recursive=True))
    if csv_files:
        train_pairs, test_pairs = [], []
        for csv_path in csv_files:
            csv_dir = os.path.dirname(csv_path)
            with open(csv_path, 'r') as f:
                rows = list(csv.reader(f))
            pairs = []
            for row in rows:
                if len(row) < 2: continue
                rgb_rel = row[0].strip().lstrip('/')
                dep_rel = row[1].strip().lstrip('/')
                for base in (csv_dir, data_root, os.path.dirname(csv_dir)):
                    rp = os.path.join(base, rgb_rel); dp = os.path.join(base, dep_rel)
                    if os.path.exists(rp) and os.path.exists(dp):
                        pairs.append((rp, dp)); break
                    parts_r = rgb_rel.split('/', 1); parts_d = dep_rel.split('/', 1)
                    if len(parts_r) > 1:
                        rp2 = os.path.join(base, parts_r[1]); dp2 = os.path.join(base, parts_d[1])
                        if os.path.exists(rp2) and os.path.exists(dp2):
                            pairs.append((rp2, dp2)); break
            name = os.path.basename(csv_path).lower()
            (test_pairs if 'test' in name else train_pairs).__iadd__([])
            if 'test' in name:
                test_pairs = pairs
            else:
                train_pairs = pairs
            print(f'  {os.path.basename(csv_path)}: {len(rows)} rows -> {len(pairs)} resolved pairs')
        if train_pairs:
            return train_pairs, test_pairs
    raise RuntimeError(f'No NYU pairs discovered in {data_root}.')


In [10]:
class NYUDepthDataset(Dataset):
    """Task 1 baseline dataset; reused unchanged."""
    def __init__(self, rgb_paths, depth_paths, processor, input_size=INPUT_SIZE,
                 augment=False, keep_original=False):
        assert len(rgb_paths) == len(depth_paths)
        self.rgb_paths = rgb_paths
        self.depth_paths = depth_paths
        self.processor = processor
        self.input_size = input_size
        self.augment = augment
        self.keep_original = keep_original
    def __len__(self):
        return len(self.rgb_paths)
    def __getitem__(self, idx):
        image = Image.open(self.rgb_paths[idx]).convert('RGB')
        depth_np = load_depth_map(self.depth_paths[idx])
        if self.augment and random.random() > 0.5:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
            depth_np = np.fliplr(depth_np).copy()
        image_square = image.resize((self.input_size, self.input_size), Image.BICUBIC)
        inputs = self.processor(images=image_square, return_tensors='pt',
                                do_resize=False, do_rescale=True, do_normalize=True)
        pixel_values = inputs['pixel_values'].squeeze(0)
        d_t = torch.from_numpy(depth_np).unsqueeze(0).unsqueeze(0).float()
        d_resized = F.interpolate(d_t, size=(self.input_size, self.input_size),
                                  mode='nearest').squeeze(0)
        valid_mask = (d_resized > MIN_DEPTH) & (d_resized < MAX_DEPTH)
        out = {'pixel_values': pixel_values, 'depth_gt': d_resized, 'valid_mask': valid_mask}
        if self.keep_original:
            out['depth_original'] = torch.from_numpy(depth_np).unsqueeze(0).float()
            img_canon = image.resize((NYU_WIDTH, NYU_HEIGHT), Image.BICUBIC)
            out['rgb_original'] = torch.from_numpy(np.array(img_canon)).permute(2, 0, 1)
        return out


In [11]:
@torch.no_grad()
def validate(model, loader, loss_fn, device, eigen_crop=True):
    """Clean-eval path from Task 1. Computes 8 metrics on Eigen crop."""
    model.eval()
    total_loss = 0.0
    agg = defaultdict(float)
    n = 0
    for batch in tqdm(loader, desc='Val', leave=False):
        pv = batch['pixel_values'].to(device, non_blocking=True)
        gt_orig = batch['depth_original'].to(device, non_blocking=True)
        out = model(pixel_values=pv)
        pred = out.predicted_depth.unsqueeze(1)
        pred_480 = F.interpolate(pred, size=(NYU_HEIGHT, NYU_WIDTH),
                                 mode='bilinear', align_corners=False).squeeze(1)
        gt_480 = gt_orig.squeeze(1)
        gt_at_in = F.interpolate(gt_orig, size=pred.shape[-2:], mode='nearest').squeeze(1)
        vm_at_in = (gt_at_in > MIN_DEPTH) & (gt_at_in < MAX_DEPTH)
        loss = loss_fn(out.predicted_depth, gt_at_in, vm_at_in)
        if torch.isfinite(loss):
            total_loss += loss.item()
        m = compute_depth_metrics(pred_480, gt_480, eigen_crop=eigen_crop)
        for k, v in m.items():
            agg[k] += v
        n += 1
    return (total_loss / max(n, 1)), {k: v / max(n, 1) for k, v in agg.items()}


In [ ]:

_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def imagenet_normalize(x: torch.Tensor) -> torch.Tensor:
    m = _IMAGENET_MEAN.to(x.device, x.dtype)
    s = _IMAGENET_STD.to(x.device, x.dtype)
    return (x - m) / s
def imagenet_denormalize(x: torch.Tensor) -> torch.Tensor:
    m = _IMAGENET_MEAN.to(x.device, x.dtype)
    s = _IMAGENET_STD.to(x.device, x.dtype)
    return x * s + m

In [ ]:
def tv_loss(patch: torch.Tensor) -> torch.Tensor:
    """Anisotropic Total Variation. patch: [3, H, W] or [B, 3, H, W]."""
    dh = (patch[..., 1:, :] - patch[..., :-1, :]).abs().mean()
    dw = (patch[..., :, 1:] - patch[..., :, :-1]).abs().mean()
    return dh + dw
_p_const = torch.full((3, PATCH_SIZE, PATCH_SIZE), 0.5)
_p_check = torch.zeros(3, PATCH_SIZE, PATCH_SIZE)
_p_check[:, ::2, ::2] = 1.0; _p_check[:, 1::2, 1::2] = 1.0
assert tv_loss(_p_const).item() < 1e-6, 'T3a: tv_loss on constant must be ~0'
assert tv_loss(_p_check).item() > 0.5, 'T3b: tv_loss on checkerboard must be > 0.5'
print(f'T3 PASS: tv_const={tv_loss(_p_const).item():.2e}  tv_check={tv_loss(_p_check).item():.4f}')

T3 PASS: tv_const=0.00e+00  tv_check=2.0000


In [ ]:
def nps_loss(patch: torch.Tensor, palette: torch.Tensor) -> torch.Tensor:
    p = patch.permute(1, 2, 0).reshape(-1, 3)
    pal = palette.to(p.device, p.dtype)
    d = torch.cdist(p.unsqueeze(0), pal.unsqueeze(0)).squeeze(0)
    return d.min(dim=1).values.mean()
_pal = PRINTABLE_COLORS
_c = _pal[0]
_p_mono = _c.view(3, 1, 1).expand(3, PATCH_SIZE, PATCH_SIZE).contiguous()
_p_rand = torch.rand(3, PATCH_SIZE, PATCH_SIZE)
assert nps_loss(_p_mono, _pal).item() < 1e-5, 'T2a: nps on pure palette color must be ~0'
assert nps_loss(_p_rand, _pal).item() > 0.0, 'T2b: nps on random patch must be > 0'
print(f'T2 PASS: nps_mono={nps_loss(_p_mono,_pal).item():.2e}  nps_rand={nps_loss(_p_rand,_pal).item():.4f}')


T2 PASS: nps_mono=0.00e+00  nps_rand=0.2377


In [ ]:
def sample_eot_params(batch_size: int, device: torch.device) -> dict:
    d = device
    return {
        'theta_deg':  (torch.rand(batch_size, device=d) * 2 - 1) * EOT_ROT_DEG,
        'scale':       torch.rand(batch_size, device=d) * (EOT_SCALE_MAX - EOT_SCALE_MIN) + EOT_SCALE_MIN,
        'tx':         (torch.rand(batch_size, device=d) * 2 - 1) * EOT_TRANS_FRAC,
        'ty':         (torch.rand(batch_size, device=d) * 2 - 1) * EOT_TRANS_FRAC,
        'brightness': (torch.rand(batch_size, device=d) * 2 - 1) * EOT_BRIGHT,
        'contrast':    torch.rand(batch_size, device=d) * (EOT_CONTRAST_MAX - EOT_CONTRAST_MIN) + EOT_CONTRAST_MIN,
        'noise_std':  torch.tensor(EOT_NOISE_STD, device=d),
    }
def identity_eot(batch_size: int, device: torch.device) -> dict:
    """Null transform: no rotation, scale=1, no translation, no jitter, no noise."""
    d = device
    z = torch.zeros(batch_size, device=d)
    return {
        'theta_deg': z, 'scale': torch.ones(batch_size, device=d),
        'tx': z.clone(), 'ty': z.clone(),
        'brightness': z.clone(), 'contrast': torch.ones(batch_size, device=d),
        'noise_std': torch.tensor(0.0, device=d),
    }

In [ ]:
def build_affine_theta(scale: torch.Tensor, theta_deg: torch.Tensor,
                       tx: torch.Tensor, ty: torch.Tensor,
                       patch_hw: int, image_hw: int) -> torch.Tensor:
    B = scale.shape[0]
    theta_rad = theta_deg * (torch.pi / 180.0)
    cos = torch.cos(-theta_rad)
    sin = torch.sin(-theta_rad)
    inv_s = 1.0 / scale
    a11 =  inv_s * cos
    a12 = -inv_s * sin
    a21 =  inv_s * sin
    a22 =  inv_s * cos
    tx_in = -(a11 * tx + a12 * ty)
    ty_in = -(a21 * tx + a22 * ty)
    theta = torch.stack([
        torch.stack([a11, a12, tx_in], dim=-1),
        torch.stack([a21, a22, ty_in], dim=-1),
    ], dim=-2)
    return theta
def _pad_patch_to_image(patch: torch.Tensor, image_hw: int, batch_size: int) -> torch.Tensor:
    C, P, _ = patch.shape
    pad_total = image_hw - P
    assert pad_total >= 0, f'patch P={P} must be <= image_hw={image_hw}'
    pl = pad_total // 2
    pr = pad_total - pl
    padded = F.pad(patch, (pl, pr, pl, pr), mode='constant', value=0.0)
    return padded.unsqueeze(0).expand(batch_size, -1, -1, -1)
def differentiable_affine_composite(image: torch.Tensor,
                                    patch: torch.Tensor,
                                    eot: dict) -> tuple:
    B, _, H, W = image.shape
    assert H == W, 'current implementation assumes square image canvas'
    alpha = torch.ones(1, patch.shape[1], patch.shape[2], device=patch.device, dtype=patch.dtype)
    rgba = torch.cat([patch, alpha], dim=0)
    canvas = _pad_patch_to_image(rgba, H, B)
    theta = build_affine_theta(eot['scale'], eot['theta_deg'], eot['tx'], eot['ty'],
                               patch_hw=patch.shape[1], image_hw=H).to(image.device, image.dtype)
    grid = F.affine_grid(theta, size=[B, 4, H, W], align_corners=False)
    warped = F.grid_sample(canvas, grid, mode='bilinear',
                           padding_mode='zeros', align_corners=False)
    rgb = warped[:, :3]
    a   = warped[:, 3:4]
    comp = a * rgb + (1.0 - a) * image
    mask = (a > 0.5).float()
    return comp, mask
def dilate_mask(mask: torch.Tensor, px: int) -> torch.Tensor:
    if px <= 0:
        return mask
    k = 2 * px + 1
    return F.max_pool2d(mask, kernel_size=k, stride=1, padding=px)


In [ ]:
def apply_color_jitter(images: torch.Tensor, brightness: torch.Tensor, contrast: torch.Tensor) -> torch.Tensor:
    b = brightness.view(-1, 1, 1, 1).to(images.dtype)
    c = contrast.view(-1, 1, 1, 1).to(images.dtype)
    out = (images - 0.5) * c + 0.5 + b
    return out.clamp(0.0, 1.0)
def adv_depth_loss(pred_depth: torch.Tensor, mask: torch.Tensor,
                   target: float = MAX_DEPTH, eps: float = ADV_LOSS_EPS) -> torch.Tensor:
    if mask.dim() == 4:
        mask = mask.squeeze(1)
    m = mask.float()
    arg = (target - pred_depth + eps).clamp_min(eps)
    return (torch.log(arg) * m).sum() / m.sum().clamp_min(1.0)
def render_patch_png(patch: torch.Tensor, path: str) -> None:
    save_image(patch.detach().cpu().clamp(0, 1), path)
import math as _math_t4
_pred = torch.full((2, INPUT_SIZE, INPUT_SIZE), TARGET_DEPTH)
_mask = torch.zeros(2, 1, INPUT_SIZE, INPUT_SIZE)
_mask[:, :, 100:200, 100:200] = 1.0
_l_min = adv_depth_loss(_pred, _mask, TARGET_DEPTH).item()
assert abs(_l_min - _math_t4.log(ADV_LOSS_EPS)) < 1e-3, f'T4a: pred=target should give log(eps)={_math_t4.log(ADV_LOSS_EPS):.3f}, got {_l_min}'
_pred0 = torch.zeros(2, INPUT_SIZE, INPUT_SIZE)
_l_max = adv_depth_loss(_pred0, _mask, TARGET_DEPTH).item()
assert abs(_l_max - _math_t4.log(TARGET_DEPTH + ADV_LOSS_EPS)) < 1e-3, f'T4b: pred=0 should give log(target+eps), got {_l_max}'
_pred_mid = torch.full((2, INPUT_SIZE, INPUT_SIZE), 5.0)
_l_mid = adv_depth_loss(_pred_mid, _mask, TARGET_DEPTH).item()
assert _l_max > _l_mid > _l_min, 'T4c: loss must be monotone decreasing in pred'
print(f'T4 PASS: L(pred=0)={_l_max:.3f}  L(pred=5)={_l_mid:.3f}  L(pred=target)={_l_min:.3f}')

T4 PASS: L(pred=0)=2.303  L(pred=5)=1.610  L(pred=target)=-6.908


In [ ]:
def load_baseline_model(ckpt_path: str, device: torch.device) -> nn.Module:
    """Build HF DA-V2-S, load Task 1 checkpoint strict=True, freeze all params, eval mode."""
    model = AutoModelForDepthEstimation.from_pretrained(MODEL_CHECKPOINT).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=True)
    assert len(missing) == 0 and len(unexpected) == 0, f'strict load failed: {missing} {unexpected}'
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    print(f'Loaded baseline from {ckpt_path}')
    print(f'  ckpt epoch: {ckpt.get("epoch")}')
    print(f'  ckpt metrics: {ckpt.get("metrics")}')
    return model

In [ ]:
@torch.no_grad()
def eval_with_patch(model, loader, patch, eot_mode='identity', seed=0):
    assert eot_mode in ('identity', 'random')
    if eot_mode == 'random':
        gen = torch.Generator(device=DEVICE).manual_seed(seed)
    model.eval()
    patch_c = patch.detach().clamp(0, 1).to(DEVICE)
    agg = defaultdict(float)
    patch_stats_sum = {'mean_abs_err_vs_target': 0.0, 'frac_pred_gt_8m': 0.0,
                       'mean_pred_in_mask': 0.0}
    n = 0
    for batch in tqdm(loader, desc=f'adv[{eot_mode}]', leave=False):
        rgb = batch['rgb_original'].to(DEVICE).float() / 255.0
        rgb = F.interpolate(rgb, size=(INPUT_SIZE, INPUT_SIZE),
                            mode='bicubic', align_corners=False).clamp_(0, 1)
        B = rgb.size(0)
        if eot_mode == 'identity':
            eot = identity_eot(B, DEVICE)
        else:
            def _u(shape, lo, hi):
                return torch.rand(*shape, generator=gen, device=DEVICE) * (hi - lo) + lo
            eot = {
                'theta_deg': _u((B,), -EOT_ROT_DEG, EOT_ROT_DEG),
                'scale':     _u((B,), EOT_SCALE_MIN, EOT_SCALE_MAX),
                'tx':        _u((B,), -EOT_TRANS_FRAC, EOT_TRANS_FRAC),
                'ty':        _u((B,), -EOT_TRANS_FRAC, EOT_TRANS_FRAC),
                'brightness':_u((B,), -EOT_BRIGHT, EOT_BRIGHT),
                'contrast':  _u((B,), EOT_CONTRAST_MIN, EOT_CONTRAST_MAX),
                'noise_std': torch.tensor(EOT_NOISE_STD, device=DEVICE),
            }
        comp, mask = differentiable_affine_composite(rgb, patch_c, eot)
        if eot_mode == 'random':
            comp = apply_color_jitter(comp, eot['brightness'], eot['contrast'])
            comp = (comp + eot['noise_std'] * torch.randn(comp.shape, generator=gen, device=DEVICE)).clamp_(0, 1)
        x = imagenet_normalize(comp)
        pred = model(pixel_values=x).predicted_depth.unsqueeze(1)
        pred_480 = F.interpolate(pred, size=(NYU_HEIGHT, NYU_WIDTH),
                                 mode='bilinear', align_corners=False).squeeze(1)
        gt_480 = batch['depth_original'].to(DEVICE).squeeze(1)
        m = compute_depth_metrics(pred_480, gt_480, eigen_crop=True)
        for k, v in m.items():
            agg[k] += v
        p518 = pred.squeeze(1)
        mk  = mask.squeeze(1)
        mk_sum = mk.sum().clamp_min(1)
        patch_stats_sum['mean_pred_in_mask']         += ((p518 * mk).sum() / mk_sum).item()
        patch_stats_sum['mean_abs_err_vs_target']    += (((p518 - TARGET_DEPTH).abs() * mk).sum() / mk_sum).item()
        patch_stats_sum['frac_pred_gt_8m']           += (((p518 > 8.0).float() * mk).sum() / mk_sum).item()
        n += 1
    metrics = {k: v / max(n, 1) for k, v in agg.items()}
    patch_stats = {k: v / max(n, 1) for k, v in patch_stats_sum.items()}
    return metrics, patch_stats


In [ ]:
def _to_tensor(x, B, device):
    return torch.full((B,), float(x), device=device)
def make_eot_swept(condition: str, value: float):
    valid = set(SWEEP_GRID.keys())
    assert condition in valid, f'unknown condition {condition!r}; expected one of {valid}'
    def _factory(B: int, device):
        eot = identity_eot(B, device)
        if condition == 'rotation':
            eot['theta_deg'] = _to_tensor(value, B, device)
        elif condition == 'scale':
            eot['scale'] = _to_tensor(value, B, device)
        elif condition == 'translation':
            eot['tx'] = _to_tensor(value, B, device)
            eot['ty'] = _to_tensor(value, B, device)
        elif condition == 'brightness':
            eot['brightness'] = _to_tensor(value, B, device)
        elif condition == 'contrast':
            eot['contrast'] = _to_tensor(value, B, device)
        elif condition == 'noise_std':
            eot['noise_std'] = torch.tensor(float(value), device=device)
        return eot
    return _factory
def make_eot_position(tx: float, ty: float):
    def _factory(B: int, device):
        eot = identity_eot(B, device)
        eot['tx'] = _to_tensor(tx, B, device)
        eot['ty'] = _to_tensor(ty, B, device)
        return eot
    return _factory
def make_eot_identity():
    return lambda B, device: identity_eot(B, device)


In [ ]:
@torch.no_grad()
def eval_with_patch_custom(model, loader, patch, make_eot, apply_jitter: bool = False,
                           apply_noise: bool = False, desc: str = 'adv'):
    model.eval()
    patch_c = patch.detach().clamp(0, 1).to(DEVICE)
    agg = defaultdict(float)
    patch_stats_sum = {'mean_abs_err_vs_target': 0.0, 'frac_pred_gt_8m': 0.0,
                       'mean_pred_in_mask': 0.0}
    per_image = []
    n = 0
    img_idx_counter = 0
    for batch in tqdm(loader, desc=desc, leave=False):
        rgb = batch['rgb_original'].to(DEVICE).float() / 255.0
        rgb = F.interpolate(rgb, size=(INPUT_SIZE, INPUT_SIZE),
                            mode='bicubic', align_corners=False).clamp_(0, 1)
        B = rgb.size(0)
        eot = make_eot(B, DEVICE)
        comp, mask = differentiable_affine_composite(rgb, patch_c, eot)
        if apply_jitter:
            comp = apply_color_jitter(comp, eot['brightness'], eot['contrast'])
        if apply_noise:
            comp = (comp + eot['noise_std'] * torch.randn_like(comp)).clamp_(0, 1)
        x = imagenet_normalize(comp)
        pred = model(pixel_values=x).predicted_depth.unsqueeze(1)
        pred_480 = F.interpolate(pred, size=(NYU_HEIGHT, NYU_WIDTH),
                                 mode='bilinear', align_corners=False).squeeze(1)
        gt_480 = batch['depth_original'].to(DEVICE).squeeze(1)
        m = compute_depth_metrics(pred_480, gt_480, eigen_crop=True)
        for k, v in m.items():
            agg[k] += v
        p518 = pred.squeeze(1)
        mk   = mask.squeeze(1)
        mk_sum = mk.sum().clamp_min(1)
        patch_stats_sum['mean_pred_in_mask']      += ((p518 * mk).sum() / mk_sum).item()
        patch_stats_sum['mean_abs_err_vs_target'] += (((p518 - TARGET_DEPTH).abs() * mk).sum() / mk_sum).item()
        patch_stats_sum['frac_pred_gt_8m']        += (((p518 > 8.0).float() * mk).sum() / mk_sum).item()
        n += 1
        for b in range(B):
            mk_b  = mk[b]
            mb_sum = mk_b.sum().clamp_min(1)
            pred_b = p518[b]
            gt_b   = gt_480[b]
            valid  = (gt_b > MIN_DEPTH) & (gt_b < MAX_DEPTH)
            gt_mean = gt_b[valid].mean().item() if valid.any() else float('nan')
            per_image.append({
                'img_idx': img_idx_counter,
                'frac_pred_gt_8m':     ((pred_b > 8.0).float() * mk_b).sum().item() / mb_sum.item(),
                'mean_pred_in_mask':   ((pred_b * mk_b).sum() / mb_sum).item(),
                'gt_mean_depth':       gt_mean,
                'depth_bucket':        depth_bucket_from_value(gt_mean),
            })
            img_idx_counter += 1
    metrics     = {k: v / max(n, 1) for k, v in agg.items()}
    patch_stats = {k: v / max(n, 1) for k, v in patch_stats_sum.items()}
    return metrics, patch_stats, per_image


In [ ]:
def deterministic_subset_indices(n_total: int, k: int, seed: int) -> list:
    assert k <= n_total
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(n_total, size=k, replace=False).tolist())

def scene_label_from_path(rgb_path: str) -> str:
    parent = os.path.basename(os.path.dirname(rgb_path)).lower()
    for cat in SCENE_CATEGORIES:
        if parent.startswith(cat):
            return cat
    return 'other'

def depth_bucket_from_value(mean_depth: float) -> str:
    if not np.isfinite(mean_depth):
        return 'unknown'
    for name, (lo, hi) in DEPTH_BUCKETS.items():
        if lo <= mean_depth < hi:
            return name
    if mean_depth >= list(DEPTH_BUCKETS.values())[-1][1]:
        return list(DEPTH_BUCKETS.keys())[-1]
    return 'unknown'

def depth_bucket_from_tensor(depth_gt: torch.Tensor) -> str:
    valid = (depth_gt > MIN_DEPTH) & (depth_gt < MAX_DEPTH)
    if not valid.any():
        return 'unknown'
    return depth_bucket_from_value(depth_gt[valid].mean().item())


In [ ]:
def build_scene_stratified_pairs(train_pairs, category: str, n_per: int,
                                 seed: int) -> list:
    hits = [p for p in train_pairs if scene_label_from_path(p[0]) == category]
    if len(hits) == 0:
        return []
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(hits), size=min(n_per, len(hits)), replace=False)
    return [hits[i] for i in sorted(idx.tolist())]


In [ ]:
def run_condition_sweep(model, loader, patch, condition: str,
                        values: list, apply_jitter: bool = False,
                        apply_noise: bool = False) -> list:
    rows = []
    for v in values:
        eot = make_eot_swept(condition, v)
        aj = apply_jitter or condition in ('brightness', 'contrast')
        an = apply_noise  or condition == 'noise_std'
        metrics, patch_region, _ = eval_with_patch_custom(
            model, loader, patch, eot, apply_jitter=aj, apply_noise=an,
            desc=f'{condition}={v}')
        rows.append({'condition': condition, 'value': float(v),
                     'metrics': metrics, 'patch_region': patch_region})
    return rows

def run_position_sweep(model, loader, patch, offsets: list) -> np.ndarray:
    N = len(offsets)
    arr = np.zeros((N, N, 2), dtype=np.float32)
    for iy, ty in enumerate(offsets):
        for ix, tx in enumerate(offsets):
            eot = make_eot_position(tx, ty)
            _, patch_region, _ = eval_with_patch_custom(
                model, loader, patch, eot, desc=f'pos(tx={tx:+.2f},ty={ty:+.2f})')
            arr[iy, ix, 0] = patch_region['frac_pred_gt_8m']
            arr[iy, ix, 1] = patch_region['mean_pred_in_mask']
    return arr


In [ ]:
def run_per_scene(model, patch, train_pairs, processor, categories: list,
                  n_per: int, seed: int) -> dict:
    results = {}
    for i, cat in enumerate(categories):
        pairs = build_scene_stratified_pairs(train_pairs, cat, n_per, seed + i)
        if len(pairs) == 0:
            print(f'  {cat}: 0 images (skipping)')
            results[cat] = {'metrics': None, 'patch_region': None, 'n': 0}
            continue
        rgb = [p[0] for p in pairs]
        dep = [p[1] for p in pairs]
        ds = NYUDepthDataset(rgb, dep, processor, keep_original=True)
        ld = DataLoader(ds, batch_size=4, shuffle=False, num_workers=0, pin_memory=True)
        metrics, patch_region, _ = eval_with_patch_custom(
            model, ld, patch, make_eot_identity(), desc=f'scene={cat}')
        results[cat] = {'metrics': metrics, 'patch_region': patch_region, 'n': len(pairs)}
        print(f'  {cat:14s} n={len(pairs):3d}  success={patch_region["frac_pred_gt_8m"]:.3f}  '
              f'mean_pred={patch_region["mean_pred_in_mask"]:.2f}m')
    return results
def run_depth_bucket(model, loader, patch) -> dict:
    _, _, per_img = eval_with_patch_custom(model, loader, patch, make_eot_identity(),
                                           desc='depth_bucket')
    buckets = {name: {'frac_pred_gt_8m': [], 'mean_pred_in_mask': [],
                      'gt_mean_depth': [], 'n': 0}
               for name in DEPTH_BUCKETS}
    for row in per_img:
        b = row['depth_bucket']
        if b in buckets:
            buckets[b]['frac_pred_gt_8m'].append(row['frac_pred_gt_8m'])
            buckets[b]['mean_pred_in_mask'].append(row['mean_pred_in_mask'])
            buckets[b]['gt_mean_depth'].append(row['gt_mean_depth'])
            buckets[b]['n'] += 1
    out = {}
    for b, d in buckets.items():
        out[b] = {
            'n':                   d['n'],
            'frac_pred_gt_8m':     float(np.mean(d['frac_pred_gt_8m'])) if d['n'] else 0.0,
            'mean_pred_in_mask':   float(np.mean(d['mean_pred_in_mask'])) if d['n'] else 0.0,
            'gt_mean_depth':       float(np.mean(d['gt_mean_depth']))    if d['n'] else 0.0,
        }
    return out


In [ ]:
def plot_sweep(rows: list, condition: str, save_path: str):
    xs = [r['value'] for r in rows]
    y1 = [r['patch_region']['frac_pred_gt_8m']        for r in rows]
    y2 = [r['patch_region']['mean_abs_err_vs_target'] for r in rows]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(xs, y1, 'o-', lw=2)
    axes[0].set_xlabel(condition); axes[0].set_ylabel('frac_pred_gt_8m')
    axes[0].set_title(f'{condition}: attack success rate')
    axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1.0)
    axes[1].plot(xs, y2, 's-', lw=2, color='C1')
    axes[1].set_xlabel(condition); axes[1].set_ylabel('mean |pred - target| (m)')
    axes[1].set_title(f'{condition}: |pred - {TARGET_DEPTH:.0f}m|')
    axes[1].grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(save_path, dpi=110); plt.close(fig)
def plot_summary_figure(all_sweeps: dict, save_path: str):
    conds = list(all_sweeps.keys())
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    for ax, cond in zip(axes.flat, conds):
        rows = all_sweeps[cond]
        xs = [r['value'] for r in rows]
        ys = [r['patch_region']['frac_pred_gt_8m'] for r in rows]
        ax.plot(xs, ys, 'o-', lw=2)
        ax.set_xlabel(cond); ax.set_ylabel('success rate')
        ax.set_title(cond); ax.set_ylim(0, 1.0); ax.grid(alpha=0.3)
    fig.suptitle('V2 attack success rate across conditions (identity EoT otherwise)',
                 fontsize=13)
    fig.tight_layout(); fig.savefig(save_path, dpi=110); plt.close(fig)
def plot_scene_bars(scene_results: dict, save_path: str):
    cats = [c for c, d in scene_results.items() if d['n'] > 0]
    vals = [scene_results[c]['patch_region']['frac_pred_gt_8m'] for c in cats]
    ns   = [scene_results[c]['n'] for c in cats]
    fig, ax = plt.subplots(figsize=(8, 3.5))
    bars = ax.bar(cats, vals, color='C2')
    for b, v, n in zip(bars, vals, ns):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.2f}\nn={n}',
                ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, max(vals) * 1.25 if vals else 1.0)
    ax.set_ylabel('frac_pred_gt_8m (attack success)')
    ax.set_title('Per-scene attack effectiveness (v2, identity EoT)')
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout(); fig.savefig(save_path, dpi=110); plt.close(fig)
def plot_bucket_bars(bucket_results: dict, save_path: str):
    buckets = list(bucket_results.keys())
    vals    = [bucket_results[b]['frac_pred_gt_8m'] for b in buckets]
    ns      = [bucket_results[b]['n']              for b in buckets]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    bars = ax.bar(buckets, vals, color='C4')
    for b, v, n in zip(bars, vals, ns):
        ax.text(b.get_x() + b.get_width()/2, v + 0.01, f'{v:.2f}\nn={n}',
                ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, max(vals) * 1.25 if vals else 1.0)
    ax.set_ylabel('frac_pred_gt_8m'); ax.set_title('Per-depth-bucket attack effectiveness')
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout(); fig.savefig(save_path, dpi=110); plt.close(fig)
def plot_position_heatmap(arr: np.ndarray, offsets: list, save_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, ch, title in zip(axes, (0, 1),
                             ('frac_pred_gt_8m', 'mean_pred_in_mask (m)')):
        im = ax.imshow(arr[..., ch], cmap='viridis', origin='upper')
        ax.set_xticks(range(len(offsets))); ax.set_xticklabels([f'{o:+.1f}' for o in offsets])
        ax.set_yticks(range(len(offsets))); ax.set_yticklabels([f'{o:+.1f}' for o in offsets])
        ax.set_xlabel('tx'); ax.set_ylabel('ty'); ax.set_title(title)
        fig.colorbar(im, ax=ax, fraction=0.045)
        for iy in range(arr.shape[0]):
            for ix in range(arr.shape[1]):
                ax.text(ix, iy, f'{arr[iy,ix,ch]:.2f}', ha='center', va='center',
                        color='white' if arr[iy,ix,ch] < arr[...,ch].max()/2 else 'black', fontsize=9)
    fig.suptitle('V2 patch position heatmap (identity EoT, translation-only)')
    fig.tight_layout(); fig.savefig(save_path, dpi=110); plt.close(fig)
def plot_variant_comparison(per_patch_sweeps: dict, save_path: str):
    conds = list(next(iter(per_patch_sweeps.values())).keys())
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    for ax, cond in zip(axes.flat, conds):
        for name, sweeps in per_patch_sweeps.items():
            rows = sweeps[cond]
            xs = [r['value'] for r in rows]
            ys = [r['patch_region']['frac_pred_gt_8m'] for r in rows]
            ax.plot(xs, ys, 'o-', lw=2, label=name)
        ax.set_xlabel(cond); ax.set_ylabel('success')
        ax.set_title(cond); ax.set_ylim(0, 1.0); ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle('V2 vs tuning variants: attack success across conditions')
    fig.tight_layout(); fig.savefig(save_path, dpi=110); plt.close(fig)


In [27]:
class TestT3Utils(unittest.TestCase):
    def test_scene_label_from_path(self):
        cases = {
            r'D:\a\nyu2_train\living_room_0038_out\37.jpg':  'living_room',
            r'D:\a\nyu2_train\bedroom_0012_out\5.jpg':       'bedroom',
            r'D:\a\nyu2_train\kitchen_0003_out\0.jpg':       'kitchen',
            r'D:\a\nyu2_train\playroom_0001_out\1.jpg':      'other',
        }
        for path, expected in cases.items():
            self.assertEqual(scene_label_from_path(path), expected, msg=path)
    def test_depth_bucket_boundaries(self):
        self.assertEqual(depth_bucket_from_value(1.0),  'shallow')
        self.assertEqual(depth_bucket_from_value(1.99), 'shallow')
        self.assertEqual(depth_bucket_from_value(2.0),  'mid')
        self.assertEqual(depth_bucket_from_value(3.99), 'mid')
        self.assertEqual(depth_bucket_from_value(4.0),  'deep')
        self.assertEqual(depth_bucket_from_value(9.99), 'deep')
        self.assertEqual(depth_bucket_from_value(float('nan')), 'unknown')
    def test_deterministic_subset_reproducible(self):
        a = deterministic_subset_indices(654, 96, 42)
        b = deterministic_subset_indices(654, 96, 42)
        self.assertEqual(a, b)
        self.assertEqual(len(a), 96)
        self.assertEqual(len(set(a)), 96)
        self.assertTrue(all(0 <= i < 654 for i in a))
    def test_make_eot_swept_isolation(self):
        f = make_eot_swept('rotation', 15.0)
        e = f(4, DEVICE)
        self.assertTrue(torch.all(e['theta_deg'] == 15.0))
        self.assertTrue(torch.all(e['scale'] == 1.0))
        self.assertTrue(torch.all(e['tx'] == 0.0))
        self.assertTrue(torch.all(e['ty'] == 0.0))
        self.assertTrue(torch.all(e['brightness'] == 0.0))
        self.assertTrue(torch.all(e['contrast'] == 1.0))
    def test_make_eot_position_identity(self):
        f = make_eot_position(0.0, 0.0)
        e = f(4, DEVICE)
        ref = identity_eot(4, DEVICE)
        for k in ref:
            self.assertTrue(torch.allclose(e[k], ref[k]), msg=f'key {k} differs')
    def test_sweep_grid_values_all_covered(self):

        for cond, vals in SWEEP_GRID.items():
            f = make_eot_swept(cond, vals[0])
            e = f(2, DEVICE)
            self.assertIn('theta_deg', e)

unittest.main(argv=['_'], module='__main__', exit=False, verbosity=2)


test_attack_report_v2_exists (__main__.TestT3Globals.test_attack_report_v2_exists) ... 

ok


test_bucket_boundaries_contiguous (__main__.TestT3Globals.test_bucket_boundaries_contiguous) ... 

ok


test_position_offsets_symmetric (__main__.TestT3Globals.test_position_offsets_symmetric) ... 

...

test_scene_label_from_path (__main__.TestT3Utils.test_scene_label_from_path) ... 

ok


test_sweep_grid_values_all_covered (__main__.TestT3Utils.test_sweep_grid_values_all_covered) ... 

ok


----------------------------------------------------------------------
Ran 13 tests in 0.144s

OK


## 4. Data

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)
print(f'Processor: {type(image_processor).__name__}')
print(f'  mean={image_processor.image_mean}  std={image_processor.image_std}')

Processor: DPTImageProcessor
  mean=(0.485, 0.456, 0.406)  std=(0.229, 0.224, 0.225)


In [ ]:
train_pairs, test_pairs = discover_nyu_pairs(DATA_ROOT)
print(f'Train pairs: {len(train_pairs)}  |  Test pairs: {len(test_pairs)}')
random.seed(SEED)
if TRAIN_SUBSET_SIZE is not None and len(train_pairs) > TRAIN_SUBSET_SIZE:
    train_sub = random.sample(train_pairs, TRAIN_SUBSET_SIZE)
    print(f'Subsampled train: {len(train_pairs)} -> {len(train_sub)} pairs')
else:
    train_sub = train_pairs
train_rgb = [p[0] for p in train_sub]
train_dep = [p[1] for p in train_sub]
test_rgb  = [p[0] for p in test_pairs]
test_dep  = [p[1] for p in test_pairs]
attack_train_ds = NYUDepthDataset(train_rgb, train_dep, image_processor, augment=False, keep_original=True)
eval_test_ds    = NYUDepthDataset(test_rgb,  test_dep,  image_processor, augment=False, keep_original=True)
attack_train_loader = DataLoader(attack_train_ds, batch_size=ATTACK_BATCH, shuffle=True,
                                  num_workers=0, pin_memory=True, drop_last=True)
eval_test_loader    = DataLoader(eval_test_ds,    batch_size=4,           shuffle=False,
                                  num_workers=0, pin_memory=True)
print(f'Attack train: {len(attack_train_ds)} samples -> {len(attack_train_loader)} batches (bs={ATTACK_BATCH})')
print(f'Eval test:    {len(eval_test_ds)} samples')
_b = next(iter(attack_train_loader))
print('Batch shapes:')
for k, v in _b.items():
    print(f'  {k}: {tuple(v.shape)} {v.dtype}')

  nyu2_test.csv: 654 rows -> 654 resolved pairs


  nyu2_train.csv: 50688 rows -> 50688 resolved pairs
Train pairs: 50688  |  Test pairs: 654
Subsampled train: 50688 -> 15000 pairs
Attack train: 15000 samples -> 3750 batches (bs=4)
Eval test:    654 samples
Batch shapes:
  pixel_values: (4, 3, 518, 518) torch.float32
  depth_gt: (4, 1, 518, 518) torch.float32
  valid_mask: (4, 1, 518, 518) torch.bool
  depth_original: (4, 1, 480, 640) torch.float32
  rgb_original: (4, 3, 480, 640) torch.uint8


In [ ]:
t3_sweep_idx = deterministic_subset_indices(len(eval_test_ds), SWEEP_SUBSET_SIZE, SWEEP_SEED)
t3_cross_idx = deterministic_subset_indices(len(eval_test_ds), CROSS_SUBSET_SIZE, CROSS_SEED)
t3_sweep_ds = torch.utils.data.Subset(eval_test_ds, t3_sweep_idx)
t3_cross_ds = torch.utils.data.Subset(eval_test_ds, t3_cross_idx)
t3_sweep_loader = DataLoader(t3_sweep_ds, batch_size=4, shuffle=False,
                             num_workers=0, pin_memory=True)
t3_cross_loader = DataLoader(t3_cross_ds, batch_size=4, shuffle=False,
                             num_workers=0, pin_memory=True)
print(f'T3 sweep subset: {len(t3_sweep_ds)}/{len(eval_test_ds)} images (seed={SWEEP_SEED})')
print(f'T3 cross subset: {len(t3_cross_ds)}/{len(eval_test_ds)} images (seed={CROSS_SEED})')

T3 sweep subset: 96/654 images (seed=42)
T3 cross subset: 64/654 images (seed=43)


## 5. Network

In [ ]:
model = load_baseline_model(BEST_MODEL_PATH, DEVICE)
gr_ok = verify_gradient_flow(model, torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE))
assert gr_ok, 'Frozen model must still allow gradient flow to input pixels.'

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Loaded baseline from D:\cv\checkpoints\best_model.pth
  ckpt epoch: 2
  ckpt metrics: {'abs_rel': 0.08969896945466356, 'sq_rel': 0.0432304761361149, 'rmse': 0.34927842911423707, 'rmse_log': 0.12098649950561727, 'log10': 0.03779537297739852, 'delta1': 0.9350847332942777, 'delta2': 0.9896846119223571, 'delta3': 0.9977638990413852}


  input grad: min=1.20e-05 max=1.19e+02 mean=3.18e+00


In [ ]:
patch_v2 = torch.load(PATCH_V2_PATH, map_location=DEVICE, weights_only=True).to(DEVICE)
print(f'V2 patch: {tuple(patch_v2.shape)}  '
      f'range [{patch_v2.min().item():.3f}, {patch_v2.max().item():.3f}]')
with open(ATTACK_REPORT_V2_PATH, 'r') as f:
    _t2_report = json.load(f)
_expected_sha = _t2_report['patch']['sha256_pt']
_actual_sha = hashlib.sha256(open(PATCH_V2_PATH, 'rb').read()).hexdigest()
assert _actual_sha == _expected_sha, (
    f'Patch sha256 mismatch!\n  expected: {_expected_sha}\n  actual:   {_actual_sha}')
print(f'sha256 OK: {_actual_sha[:16]}... matches attack_report.json')
with open(BASELINE_METRICS_PATH, 'r') as f:
    _baseline_full = json.load(f)
_baseline = _baseline_full.get('metrics', _baseline_full)
print(f'Baseline (Task 1 clean eval): delta1={_baseline["delta1"]:.4f}  '
      f'abs_rel={_baseline["abs_rel"]:.4f}  rmse={_baseline["rmse"]:.4f}')
print(f'Task 2 headline (v2, det):   frac_pred_gt_8m={_t2_report["adv_deterministic"]["patch_region"]["frac_pred_gt_8m"]:.4f}')
print(f'Task 2 headline (v2, rnd):   frac_pred_gt_8m={_t2_report["adv_random_eot_3seeds"]["patch_region"]["frac_pred_gt_8m"]:.4f}')


V2 patch: (3, 128, 128)  range [0.000, 1.000]
sha256 OK: 605822a9bce22eda... matches attack_report.json
Baseline (Task 1 clean eval): delta1=0.9351  abs_rel=0.0897  rmse=0.3493
Task 2 headline (v2, det):   frac_pred_gt_8m=0.4930
Task 2 headline (v2, rnd):   frac_pred_gt_8m=0.5000


## 6. Train (tuning variants)

- **V3a `v3a_large192`**: 192x192 patch
- **V3b `v3b_ladvx5`**: 128x128

In [ ]:
def attack_step(patch, batch, model, eot_on=True):
    rgb = batch['rgb_original'].to(DEVICE).float() / 255.0
    rgb = F.interpolate(rgb, size=(INPUT_SIZE, INPUT_SIZE),
                        mode='bicubic', align_corners=False).clamp_(0, 1)
    B = rgb.size(0)
    eot = sample_eot_params(B, DEVICE) if eot_on else identity_eot(B, DEVICE)
    patch_c = patch.clamp(0, 1)
    comp, mask = differentiable_affine_composite(rgb, patch_c, eot)
    if eot_on:
        comp = apply_color_jitter(comp, eot['brightness'], eot['contrast'])
        comp = (comp + eot['noise_std'] * torch.randn_like(comp)).clamp_(0, 1)
    x = imagenet_normalize(comp)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP_FORWARD):
        pred = model(pixel_values=x).predicted_depth
    pred = pred.float()
    mask_eff = dilate_mask(mask, MASK_DILATE_PX)
    L_adv = adv_depth_loss(pred, mask_eff, TARGET_DEPTH)
    L_nps = nps_loss(patch_c, PRINTABLE_COLORS.to(DEVICE))
    L_tv  = tv_loss(patch_c)
    L = LAMBDA_ADV * L_adv + LAMBDA_NPS * L_nps + LAMBDA_TV * L_tv
    meta = {
        'L_adv': L_adv.item(), 'L_nps': L_nps.item(), 'L_tv': L_tv.item(),
        'mean_pred_in_mask': ((pred * mask.squeeze(1)).sum() / mask.sum().clamp_min(1)).item(),
    }
    return L, meta


In [ ]:
def train_patch_variant(variant_name: str, cfg: dict,
                        attack_train_loader, model) -> torch.Tensor:
    out_pt  = os.path.join(CHECKPOINT_DIR, f'adv_patch_{variant_name}.pt')
    out_png = os.path.join(CHECKPOINT_DIR, f'adv_patch_{variant_name}.png')
    out_rep = os.path.join(CHECKPOINT_DIR, f'attack_report_{variant_name}.json')
    if os.path.exists(out_pt):
        print(f'[{variant_name}] Existing patch at {out_pt} -- loading and skipping train.')
        return torch.load(out_pt, map_location=DEVICE, weights_only=True).to(DEVICE)
    saved = {k: globals()[k] for k in
             ('LAMBDA_ADV','LAMBDA_NPS','LAMBDA_TV','TARGET_DEPTH','ADV_LOSS_EPS','PATCH_SIZE')}
    globals()['LAMBDA_ADV']    = cfg['lambda_adv']
    globals()['LAMBDA_NPS']    = cfg['lambda_nps']
    globals()['LAMBDA_TV']     = cfg['lambda_tv']
    globals()['TARGET_DEPTH']  = cfg['target_depth']
    globals()['ADV_LOSS_EPS']  = cfg['adv_loss_eps']
    globals()['PATCH_SIZE']    = cfg['patch_size']
    try:
        torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
        p = torch.empty(3, cfg['patch_size'], cfg['patch_size'],
                        device=DEVICE).uniform_(0, 1).requires_grad_(True)
        opt = torch.optim.Adam([p], lr=cfg['attack_lr'])
        it = iter(attack_train_loader)
        log = []
        t0 = time.time()
        print(f'[{variant_name}] train start  patch={cfg["patch_size"]}x{cfg["patch_size"]}  '
              f'lambda_adv={cfg["lambda_adv"]}  lambda_nps={cfg["lambda_nps"]}  steps={cfg["n_steps"]}')
        for step in range(cfg['n_steps']):
            try:    batch = next(it)
            except StopIteration:
                it = iter(attack_train_loader); batch = next(it)
            L, meta = attack_step(p, batch, model, eot_on=True)
            opt.zero_grad(set_to_none=True)
            L.backward()
            torch.nn.utils.clip_grad_norm_([p], max_norm=GRAD_CLIP_PATCH)
            opt.step()
            with torch.no_grad():
                p.clamp_(0, 1)
            log.append({'step': step, 'L': L.item(), **meta})
            if step % LOG_EVERY == 0 or step == cfg['n_steps'] - 1:
                elapsed = time.time() - t0
                eta = elapsed / max(step+1, 1) * (cfg['n_steps'] - step - 1)
                print(f'  [{step:5d}/{cfg["n_steps"]}] L={L.item():.4f}  '
                      f'L_adv={meta["L_adv"]:.4f}  mean_pred={meta["mean_pred_in_mask"]:.2f}m  '
                      f'elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m')
            if step % SAVE_EVERY == 0 or step == cfg['n_steps'] - 1:
                torch.save(p.detach().cpu(), out_pt)
                render_patch_png(p, out_png)
        total_min = (time.time() - t0) / 60.0
        report = {
            'variant': variant_name,
            'config':  cfg,
            'train_minutes': total_min,
            'patch_path': out_pt,
            'sha256_pt': hashlib.sha256(open(out_pt, 'rb').read()).hexdigest(),
            'final_log': log[-5:],
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        }
        with open(out_rep, 'w') as f:
            json.dump(report, f, indent=2)
        print(f'[{variant_name}] DONE in {total_min:.1f} min  ->  {out_pt}')
        return p.detach()
    finally:
        for k, v in saved.items():
            globals()[k] = v


In [ ]:
trained_variants = {}
for name, cfg in VARIANT_CONFIGS.items():
    trained_variants[name] = train_patch_variant(name, cfg, attack_train_loader, model)
print(f'\nTrained variants: {list(trained_variants.keys())}')
for name, p in trained_variants.items():
    print(f'  {name}: shape={tuple(p.shape)}  range=[{p.min().item():.3f}, {p.max().item():.3f}]')

[v3a_large192] train start  patch=192x192  lambda_adv=1.0  lambda_nps=0.005  steps=3000


  [    0/3000] L=2.0057  L_adv=2.0038  mean_pred=2.48m  elapsed=0.0m  ETA=23.6m


  [   25/3000] L=1.9782  L_adv=1.9764  mean_pred=2.67m  elapsed=0.1m  ETA=6.8m


  [   50/3000] L=1.6220  L_adv=1.6201  mean_pred=4.55m  elapsed=0.1m  ETA=6.6m


  [   75/3000] L=1.4999  L_adv=1.4981  mean_pred=5.39m  elapsed=0.2m  ETA=6.3m


...

  [ 2900/3000] L=-4.5463  L_adv=-0.9094  mean_pred=8.38m  elapsed=6.6m  ETA=0.2m


  [ 2925/3000] L=-5.5312  L_adv=-1.1064  mean_pred=8.25m  elapsed=6.7m  ETA=0.2m


  [ 2950/3000] L=-4.0513  L_adv=-0.8104  mean_pred=7.76m  elapsed=6.8m  ETA=0.1m


  [ 2975/3000] L=-1.0249  L_adv=-0.2051  mean_pred=7.60m  elapsed=6.9m  ETA=0.1m


  [ 2999/3000] L=-10.5933  L_adv=-2.1188  mean_pred=8.49m  elapsed=7.0m  ETA=0.0m
[v3b_ladvx5] DONE in 7.0 min  ->  D:\cv\checkpoints\adv_patch_v3b_ladvx5.pt

Trained variants: ['v3a_large192', 'v3b_ladvx5']
  v3a_large192: shape=(3, 192, 192)  range=[0.000, 1.000]
  v3b_ladvx5: shape=(3, 128, 128)  range=[0.000, 1.000]


## 7. Evaluation

### 7a. Clean-eval regression (no patch)

In [ ]:
_silog = SILogLoss(variance_focus=0.5)
_, _clean_metrics = validate(model, eval_test_loader, _silog, DEVICE, eigen_crop=True)
print('Clean re-eval:')
for k in ('delta1','abs_rel','rmse'):
    print(f'  {k:10s} now={_clean_metrics[k]:.6f}  baseline={_baseline[k]:.6f}  '
          f'diff={abs(_clean_metrics[k] - _baseline[k]):.2e}')
for k in _baseline:
    if k in _clean_metrics:
        assert abs(_clean_metrics[k] - _baseline[k]) < 1e-4, \
            f'7a FAIL: {k} drifted from baseline by {abs(_clean_metrics[k]-_baseline[k]):.2e}'
print('7a PASS: clean-eval reproduces baseline (all 8 metrics within 1e-4)')

Val:   0%|          | 0/164 [00:00<?, ?it/s]

Clean re-eval:
  delta1     now=0.935085  baseline=0.935085  diff=0.00e+00
  abs_rel    now=0.089699  baseline=0.089699  diff=0.00e+00
  rmse       now=0.349278  baseline=0.349278  diff=0.00e+00
7a PASS: clean-eval reproduces baseline (all 8 metrics within 1e-4)


### 7b. V2 attack regression (full 654)

In [ ]:
adv_det_metrics_v2, adv_det_ps_v2 = eval_with_patch(model, eval_test_loader, patch_v2, 'identity')
adv_rnd_metrics_v2 = _t2_report['adv_random_eot_3seeds']['metrics']
adv_rnd_ps_v2      = _t2_report['adv_random_eot_3seeds']['patch_region']
print('V2 attack regression vs attack_report.json:')
print(f'  det  frac_pred_gt_8m  now={adv_det_ps_v2["frac_pred_gt_8m"]:.6f}  '
      f'ref={_t2_report["adv_deterministic"]["patch_region"]["frac_pred_gt_8m"]:.6f}')
print(f'  rnd  frac_pred_gt_8m  now={adv_rnd_ps_v2["frac_pred_gt_8m"]:.6f}  '
      f'ref={_t2_report["adv_random_eot_3seeds"]["patch_region"]["frac_pred_gt_8m"]:.6f}')
for k in ('frac_pred_gt_8m','mean_pred_in_mask','mean_abs_err_vs_target'):
    assert abs(adv_det_ps_v2[k] - _t2_report['adv_deterministic']['patch_region'][k]) < 1e-3, \
        f'7b det {k} drift > 1e-3'
for k in ('delta1','abs_rel','rmse'):
    assert abs(adv_det_metrics_v2[k] - _t2_report['adv_deterministic']['metrics'][k]) < 1e-3, \
        f'7b det {k} drift > 1e-3'
print('7b PASS: v2 attack numbers reproduce attack_report.json (tolerance 1e-3)')

adv[identity]:   0%|          | 0/164 [00:00<?, ?it/s]

V2 attack regression vs attack_report.json:
  det  frac_pred_gt_8m  now=0.493023  ref=0.493023
  rnd  frac_pred_gt_8m  now=0.499973  ref=0.499973
7b PASS: v2 attack numbers reproduce attack_report.json (tolerance 1e-3)


### 7c. EoT condition sweeps (V2 patch, 96-image subset)

In [38]:
v2_sweeps = {}
for cond, values in SWEEP_GRID.items():
    print(f'Sweep: {cond}  values={values}')
    rows = run_condition_sweep(model, t3_sweep_loader, patch_v2, cond, values)
    v2_sweeps[cond] = rows
    plot_sweep(rows, cond, os.path.join(T3_VIZ_DIR, f'sweep_{cond}.png'))
    for r in rows:
        print(f'  {cond:12s}={r["value"]:+.3f}  '
              f'frac_gt_8m={r["patch_region"]["frac_pred_gt_8m"]:.3f}  '
              f'mean_pred={r["patch_region"]["mean_pred_in_mask"]:.2f}m')
print(f'Saved: {T3_VIZ_DIR}\\sweep_*.png ({len(v2_sweeps)} files)')


Sweep: rotation  values=[-30, -15, -5, 0, 5, 15, 30]


rotation=-30:   0%|          | 0/24 [00:00<?, ?it/s]

rotation=-15:   0%|          | 0/24 [00:00<?, ?it/s]

rotation=-5:   0%|          | 0/24 [00:00<?, ?it/s]

rotation=0:   0%|          | 0/24 [00:00<?, ?it/s]

...

noise_std=0.0:   0%|          | 0/24 [00:00<?, ?it/s]

noise_std=0.02:   0%|          | 0/24 [00:00<?, ?it/s]

noise_std=0.05:   0%|          | 0/24 [00:00<?, ?it/s]

noise_std=0.1:   0%|          | 0/24 [00:00<?, ?it/s]

  noise_std   =+0.000  frac_gt_8m=0.522  mean_pred=7.49m
  noise_std   =+0.020  frac_gt_8m=0.568  mean_pred=7.70m
  noise_std   =+0.050  frac_gt_8m=0.580  mean_pred=7.77m
  noise_std   =+0.100  frac_gt_8m=0.474  mean_pred=7.29m
Saved: D:\cv\checkpoints\viz_task3\sweep_*.png (6 files)


### 7d. Patch-position heatmap (V2 patch, 3x3 grid)

In [ ]:
v2_position = run_position_sweep(model, t3_sweep_loader, patch_v2, POSITION_OFFSETS)
plot_position_heatmap(v2_position, POSITION_OFFSETS,
                      os.path.join(T3_VIZ_DIR, 'position_heatmap.png'))
print(f'Position heatmap (success-rate grid, tx=cols, ty=rows):')
for iy, ty in enumerate(POSITION_OFFSETS):
    row = '  '.join(f'{v2_position[iy, ix, 0]:.3f}' for ix in range(len(POSITION_OFFSETS)))
    print(f'  ty={ty:+.1f}  {row}')
print(f'Saved: {T3_VIZ_DIR}\\position_heatmap.png')

pos(tx=-0.30,ty=-0.30):   0%|          | 0/24 [00:00<?, ?it/s]

pos(tx=+0.00,ty=-0.30):   0%|          | 0/24 [00:00<?, ?it/s]

pos(tx=+0.30,ty=-0.30):   0%|          | 0/24 [00:00<?, ?it/s]

pos(tx=-0.30,ty=+0.00):   0%|          | 0/24 [00:00<?, ?it/s]

pos(tx=+0.00,ty=+0.00):   0%|          | 0/24 [00:00<?, ?it/s]

pos(tx=+0.30,ty=+0.00):   0%|          | 0/24 [00:00<?, ?it/s]

pos(tx=-0.30,ty=+0.30):   0%|          | 0/24 [00:00<?, ?it/s]

pos(tx=+0.00,ty=+0.30):   0%|          | 0/24 [00:00<?, ?it/s]

pos(tx=+0.30,ty=+0.30):   0%|          | 0/24 [00:00<?, ?it/s]

Position heatmap (success-rate grid, tx=cols, ty=rows):
  ty=-0.3  0.644  0.687  0.661
  ty=+0.0  0.440  0.522  0.423
  ty=+0.3  0.268  0.236  0.287
Saved: D:\cv\checkpoints\viz_task3\position_heatmap.png


### 7e. Per-scene characterization (V2 patch, train subsets)

In [ ]:
print('Per-scene characterization (identity EoT):')
v2_per_scene = run_per_scene(model, patch_v2, train_pairs, image_processor,
                             SCENE_CATEGORIES, PER_SCENE_SAMPLES, PER_SCENE_SEED)
plot_scene_bars(v2_per_scene, os.path.join(T3_VIZ_DIR, 'per_scene.png'))
print(f'Saved: {T3_VIZ_DIR}\\per_scene.png')

Per-scene characterization (identity EoT):


scene=bedroom:   0%|          | 0/25 [00:00<?, ?it/s]

  bedroom        n=100  success=0.498  mean_pred=7.36m


scene=living_room:   0%|          | 0/25 [00:00<?, ?it/s]

  living_room    n=100  success=0.535  mean_pred=7.55m


...

  bathroom       n=100  success=0.486  mean_pred=7.12m


scene=office:   0%|          | 0/25 [00:00<?, ?it/s]

  office         n=100  success=0.517  mean_pred=7.45m


scene=dining_room:   0%|          | 0/25 [00:00<?, ?it/s]

  dining_room    n=100  success=0.550  mean_pred=7.61m
Saved: D:\cv\checkpoints\viz_task3\per_scene.png


### 7f. Depth-bucket stratification (V2 patch, full 654)

In [ ]:
v2_buckets = run_depth_bucket(model, eval_test_loader, patch_v2)
print('Per-depth-bucket attack effectiveness:')
for name, d in v2_buckets.items():
    print(f'  {name:8s}  n={d["n"]:4d}  gt_mean={d["gt_mean_depth"]:.2f}m  '
          f'frac_gt_8m={d["frac_pred_gt_8m"]:.3f}  mean_pred={d["mean_pred_in_mask"]:.2f}m')
plot_bucket_bars(v2_buckets, os.path.join(T3_VIZ_DIR, 'depth_buckets.png'))
print(f'Saved: {T3_VIZ_DIR}\\depth_buckets.png')

depth_bucket:   0%|          | 0/164 [00:00<?, ?it/s]

Per-depth-bucket attack effectiveness:
  shallow   n= 116  gt_mean=1.71m  frac_gt_8m=0.469  mean_pred=6.95m
  mid       n= 499  gt_mean=2.84m  frac_gt_8m=0.488  mean_pred=7.34m
  deep      n=  39  gt_mean=4.51m  frac_gt_8m=0.653  mean_pred=8.16m
Saved: D:\cv\checkpoints\viz_task3\depth_buckets.png


### 7g. Cross-patch sweeps (V3a, V3b on 64-image subset)

In [ ]:
cross_sweeps = {'v2': {}}
for cond, values in SWEEP_GRID.items():
    cross_sweeps['v2'][cond] = run_condition_sweep(model, t3_cross_loader, patch_v2, cond, values)
for name, p in trained_variants.items():
    print(f'Cross sweeps for {name}:')
    cross_sweeps[name] = {}
    for cond, values in SWEEP_GRID.items():
        rows = run_condition_sweep(model, t3_cross_loader, p, cond, values)
        cross_sweeps[name][cond] = rows
        print(f'  {name:14s} {cond:12s} final_success={rows[-1]["patch_region"]["frac_pred_gt_8m"]:.3f}')
plot_variant_comparison(cross_sweeps, os.path.join(T3_VIZ_DIR, 'variant_comparison.png'))
print(f'Saved: {T3_VIZ_DIR}\\variant_comparison.png')

rotation=-30:   0%|          | 0/16 [00:00<?, ?it/s]

rotation=-15:   0%|          | 0/16 [00:00<?, ?it/s]

rotation=-5:   0%|          | 0/16 [00:00<?, ?it/s]

rotation=0:   0%|          | 0/16 [00:00<?, ?it/s]

rotation=5:   0%|          | 0/16 [00:00<?, ?it/s]

...

noise_std=0.02:   0%|          | 0/16 [00:00<?, ?it/s]

noise_std=0.05:   0%|          | 0/16 [00:00<?, ?it/s]

noise_std=0.1:   0%|          | 0/16 [00:00<?, ?it/s]

  v3b_ladvx5     noise_std    final_success=0.475


Saved: D:\cv\checkpoints\viz_task3\variant_comparison.png


In [ ]:

variant_full_eval = {}
for name, p in trained_variants.items():
    det_m, det_ps = eval_with_patch(model, eval_test_loader, p, 'identity')
    variant_full_eval[name] = {'metrics': det_m, 'patch_region': det_ps}
    print(f'{name:14s} det  frac_gt_8m={det_ps["frac_pred_gt_8m"]:.4f}  '
          f'mean_pred={det_ps["mean_pred_in_mask"]:.3f}m  delta1={det_m["delta1"]:.4f}')
    _rp = os.path.join(CHECKPOINT_DIR, f'attack_report_{name}.json')
    if os.path.exists(_rp):
        with open(_rp, 'r') as f: _rep = json.load(f)
    else:
        _rep = {'variant': name, 'config': VARIANT_CONFIGS[name]}
    _rep['adv_deterministic'] = {'metrics': det_m, 'patch_region': det_ps}
    with open(_rp, 'w') as f: json.dump(_rep, f, indent=2)

adv[identity]:   0%|          | 0/164 [00:00<?, ?it/s]

v3a_large192   det  frac_gt_8m=0.7925  mean_pred=9.186m  delta1=0.6670


adv[identity]:   0%|          | 0/164 [00:00<?, ?it/s]

v3b_ladvx5     det  frac_gt_8m=0.4899  mean_pred=7.285m  delta1=0.8266


### 7h. Qualitative examples (top-5 success + top-5 failure)

In [ ]:
_, _, per_img_v2 = eval_with_patch_custom(model, eval_test_loader, patch_v2,
                                          make_eot_identity(), desc='rank654')
ranked = sorted(per_img_v2, key=lambda r: r['frac_pred_gt_8m'], reverse=True)
top5_success = ranked[:5]
top5_fail    = ranked[-5:]
print('Top-5 success (highest frac_pred_gt_8m):')
for r in top5_success:
    print(f'  img_idx={r["img_idx"]:3d}  frac_gt_8m={r["frac_pred_gt_8m"]:.3f}  '
          f'gt_mean={r["gt_mean_depth"]:.2f}m  bucket={r["depth_bucket"]}')
print('\nTop-5 failure (lowest frac_pred_gt_8m):')
for r in top5_fail:
    print(f'  img_idx={r["img_idx"]:3d}  frac_gt_8m={r["frac_pred_gt_8m"]:.3f}  '
          f'gt_mean={r["gt_mean_depth"]:.2f}m  bucket={r["depth_bucket"]}')


rank654:   0%|          | 0/164 [00:00<?, ?it/s]

Top-5 success (highest frac_pred_gt_8m):
  img_idx=562  frac_gt_8m=0.839  gt_mean=4.11m  bucket=deep
  img_idx=576  frac_gt_8m=0.828  gt_mean=4.24m  bucket=deep
  img_idx=596  frac_gt_8m=0.828  gt_mean=4.39m  bucket=deep
  img_idx=134  frac_gt_8m=0.817  gt_mean=5.19m  bucket=deep
  img_idx=599  frac_gt_8m=0.817  gt_mean=4.74m  bucket=deep

Top-5 failure (lowest frac_pred_gt_8m):
  img_idx=629  frac_gt_8m=0.000  gt_mean=4.11m  bucket=deep
  img_idx=630  frac_gt_8m=0.000  gt_mean=3.15m  bucket=mid
  img_idx=641  frac_gt_8m=0.000  gt_mean=2.61m  bucket=mid
  img_idx=647  frac_gt_8m=0.000  gt_mean=3.02m  bucket=mid
  img_idx=650  frac_gt_8m=0.000  gt_mean=3.23m  bucket=mid


In [ ]:
@torch.no_grad()
def render_qualitative_collage(indices: list, patch: torch.Tensor,
                               title: str, save_path: str):
    strips = []
    for idx in indices:
        sample = eval_test_ds[idx]
        rgb = (sample['rgb_original'].unsqueeze(0).to(DEVICE).float() / 255.0)
        rgb = F.interpolate(rgb, size=(INPUT_SIZE, INPUT_SIZE), mode='bicubic',
                            align_corners=False).clamp_(0, 1)
        eot = identity_eot(1, DEVICE)
        comp, mask = differentiable_affine_composite(rgb, patch.clamp(0, 1), eot)
        x_clean = imagenet_normalize(rgb)
        x_adv   = imagenet_normalize(comp)
        pred_clean = model(pixel_values=x_clean).predicted_depth[0].cpu().numpy()
        pred_adv   = model(pixel_values=x_adv  ).predicted_depth[0].cpu().numpy()
        diff = np.abs(pred_adv - pred_clean)
        fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.5))
        axes[0].imshow(comp[0].permute(1,2,0).cpu().numpy()); axes[0].set_title(f'img {idx} w/ patch')
        axes[0].axis('off')
        im1 = axes[1].imshow(pred_adv, cmap='inferno', vmin=0, vmax=10)
        axes[1].set_title('pred_adv (m)'); axes[1].axis('off')
        plt.colorbar(im1, ax=axes[1], fraction=0.045)
        im2 = axes[2].imshow(diff, cmap='magma', vmin=0, vmax=diff.max())
        axes[2].set_title('|pred_adv - pred_clean| (m)'); axes[2].axis('off')
        plt.colorbar(im2, ax=axes[2], fraction=0.045)
        fig.tight_layout()
        fig.canvas.draw()
        strip = np.array(fig.canvas.renderer.buffer_rgba())
        plt.close(fig)
        strips.append(strip)
    if not strips:
        return
    collage = np.concatenate(strips, axis=0)
    Image.fromarray(collage).save(save_path)
    print(f'Saved: {save_path}')
render_qualitative_collage([r['img_idx'] for r in top5_success], patch_v2,
                            'Top-5 success', os.path.join(T3_VIZ_DIR, 'top5_success.png'))
render_qualitative_collage([r['img_idx'] for r in top5_fail], patch_v2,
                            'Top-5 failure', os.path.join(T3_VIZ_DIR, 'top5_fail.png'))


Saved: D:\cv\checkpoints\viz_task3\top5_success.png


Saved: D:\cv\checkpoints\viz_task3\top5_fail.png


### 7i. Aggregate task3_report.json

In [46]:
task3_report = {
    'task': 'task3_characterization',
    'patches_evaluated': ['v2'] + list(VARIANT_CONFIGS.keys()),
    'subsets': {
        'sweep_96': {'size': SWEEP_SUBSET_SIZE, 'seed': SWEEP_SEED, 'indices': t3_sweep_idx},
        'cross_64': {'size': CROSS_SUBSET_SIZE, 'seed': CROSS_SEED, 'indices': t3_cross_idx},
    },
    'per_patch': {
        'v2': {
            'deterministic_full654': {'metrics': adv_det_metrics_v2, 'patch_region': adv_det_ps_v2},
            'random_eot_3seeds_full654': {'metrics': adv_rnd_metrics_v2, 'patch_region': adv_rnd_ps_v2},
            'condition_sweeps':     {c: [{'condition': r['condition'], 'value': r['value'],
                                            'metrics': r['metrics'], 'patch_region': r['patch_region']}
                                           for r in rows]
                                      for c, rows in v2_sweeps.items()},
            'position_sweep':       {'offsets': POSITION_OFFSETS,
                                     'frac_grid':      v2_position[..., 0].tolist(),
                                     'mean_pred_grid': v2_position[..., 1].tolist()},
            'per_scene':            v2_per_scene,
            'depth_buckets':        v2_buckets,
            'qualitative_picks':    {'top5_success': [r['img_idx'] for r in top5_success],
                                     'top5_fail':    [r['img_idx'] for r in top5_fail]},
        },
    },
    'cross_patch_sweeps_64subset': {name: {c: [{'condition': r['condition'],
                                                 'value': r['value'],
                                                 'metrics': r['metrics'],
                                                 'patch_region': r['patch_region']}
                                                for r in rows]
                                           for c, rows in sweeps.items()}
                                    for name, sweeps in cross_sweeps.items()},
    'variants_full654': variant_full_eval,
    'variant_configs': VARIANT_CONFIGS,
    'baseline_reference': _baseline,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}
for name in VARIANT_CONFIGS:

    task3_report['per_patch'][name] = {'deterministic_full654': variant_full_eval[name]}
    if name in cross_sweeps:
        task3_report['per_patch'][name]['condition_sweeps_64subset'] = \
            task3_report['cross_patch_sweeps_64subset'][name]
with open(T3_REPORT_PATH, 'w') as f:
    json.dump(task3_report, f, indent=2)
print(f'Saved: {T3_REPORT_PATH}')

with open(T3_REPORT_PATH) as f:
    _rt = json.load(f)
for key in ('task','patches_evaluated','subsets','per_patch','variants_full654','timestamp'):
    assert key in _rt, f'7i FAIL: missing {key}'
assert 'v2' in _rt['per_patch'], '7i FAIL: v2 missing from per_patch'
print('7i PASS: task3_report.json round-trips and has all required keys')


Saved: D:\cv\checkpoints\task3_report.json
7i PASS: task3_report.json round-trips and has all required keys


### 7j. Summary figure (2x3 grid)

In [ ]:
plot_summary_figure(v2_sweeps, os.path.join(T3_VIZ_DIR, 'summary_figure.png'))
print(f'Saved: {T3_VIZ_DIR}\\summary_figure.png')
print('\n' + '=' * 74)
print('TASK 3 HEADLINE')
print('=' * 74)
print(f'V2 patch (128x128, log-barrier):')
print(f'  full654 det  frac_pred_gt_8m = {adv_det_ps_v2["frac_pred_gt_8m"]:.4f}')
print(f'  full654 rnd  frac_pred_gt_8m = {adv_rnd_ps_v2["frac_pred_gt_8m"]:.4f}')
for name in VARIANT_CONFIGS:
    ps = variant_full_eval[name]['patch_region']
    print(f'{name} (cfg={VARIANT_CONFIGS[name]["patch_size"]}^2, lambda_adv={VARIANT_CONFIGS[name]["lambda_adv"]}):')
    print(f'  full654 det  frac_pred_gt_8m = {ps["frac_pred_gt_8m"]:.4f}  '
          f'delta={ps["frac_pred_gt_8m"] - adv_det_ps_v2["frac_pred_gt_8m"]:+.4f} vs v2')
print('=' * 74)


Saved: D:\cv\checkpoints\viz_task3\summary_figure.png

TASK 3 HEADLINE
V2 patch (128x128, log-barrier):
  full654 det  frac_pred_gt_8m = 0.4930
  full654 rnd  frac_pred_gt_8m = 0.5000
v3a_large192 (cfg=192^2, lambda_adv=1.0):
  full654 det  frac_pred_gt_8m = 0.7925  delta=+0.2995 vs v2
v3b_ladvx5 (cfg=128^2, lambda_adv=5.0):
  full654 det  frac_pred_gt_8m = 0.4899  delta=-0.0032 vs v2
